# EDA: Skin Lesions Dataset
This notebook performs an initial exploratory data analysis (EDA) on the SkinLesions dataset referenced in `configs/config.yaml`. It summarizes class counts, image size distribution, and shows example images.

In [ ]:
# Standard imports
import os
from pathlib import Path
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set(style="whitegrid")

In [ ]:
# Load config to find dataset root
cfg_path = Path('configs/config.yaml')
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)
dataset_root = Path(cfg['paths']['dataset_root'])
classes = cfg['data']['classes']
dataset_root, classes

In [ ]:
# Gather image file paths and counts per class
rows = []
for cls in classes:
    cls_dir = dataset_root / cls
    if not cls_dir.exists():
        print(f'Warning: {cls_dir} does not exist')
        continue
    imgs = [p for p in cls_dir.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']]
    for p in imgs:
        rows.append({'class': cls, 'path': str(p)})
df = pd.DataFrame(rows)
counts = df['class'].value_counts().reindex(classes).fillna(0).astype(int)
counts

In [ ]:
# Bar plot of class counts
plt.figure(figsize=(8,5))
sns.barplot(x=counts.index, y=counts.values, palette='viridis')
plt.ylabel('Number of images')
plt.title('Class distribution')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Show a grid of sample images (up to 8 per class, total limit to avoid huge outputs)
def show_samples(df, n_per_class=6, max_total=24):
    sample_paths = []
    for cls in classes:
        cls_paths = df[df['class']==cls]['path'].sample(min(n_per_class, df[df['class']==cls].shape[0])) if df[df['class']==cls].shape[0] > 0 else []
        sample_paths.extend(list(cls_paths))
    sample_paths = sample_paths[:max_total]
    n = len(sample_paths)
    cols = 4
    rows = int(np.ceil(n/cols)) if n>0 else 0
    plt.figure(figsize=(cols*3, rows*3))
    for i, p in enumerate(sample_paths):
        img = Image.open(p).convert('RGB')
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.axis('off')
        # show filename and class
        cls = Path(p).parent.name
        plt.title(f'{cls}')
    plt.tight_layout()
    plt.show()

show_samples(df)


In [ ]:
# Compute image size distribution (width x height) for a sample (limit to 200 files for speed)
sample = df['path'].sample(min(200, len(df))) if len(df)>0 else []
sizes = []
for p in sample:
    try:
        with Image.open(p) as im:
            sizes.append(im.size)  # (width, height)
    except Exception as e:
        # skip unreadable files
        continue
if len(sizes) > 0:
    widths, heights = zip(*sizes)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(widths, kde=True, bins=30)
    plt.title('Width distribution')
    plt.subplot(1,2,2)
    sns.histplot(heights, kde=True, bins=30)
    plt.title('Height distribution')
    plt.tight_layout()
    plt.show()
else:
    print('No image sizes available to summarize.')

## Next steps
- Save split manifests (train/val/test) as described in the agent handoff.
- Create a `notebooks/99_demo.ipynb` that runs a small inference demo using a saved checkpoint.
- Add more EDA: color statistics, blur/quality checks, and per-class image size filters.